# Animal Sentiment Analysis - TROVE Australian Newspapers

This notebook uses the TROVE API to analyze emotional attitudes towards animal species in Australian newspapers.

## Features:
- Query TROVE API for articles mentioning specific animal species
- Machine learning-based sentiment analysis
- Word clouds showing common terms associated with each animal
- Sentiment trend graphs over time
- Geographical heat maps showing sentiment distribution across Australian states
- Export results to CSV files

## Setup Instructions:
1. Get your TROVE API key from https://trove.nla.gov.au/
2. Insert your API key in the designated cell below
3. Specify the animal species you want to analyze
4. Run all cells to generate visualizations and analysis

In [ ]:
# ===== INSTALLATION CELL (Run this cell first) =====
# This cell installs all required packages from requirements.txt
# You only need to run this once

import sys

print("Installing all required packages...")
print("This may take 5-10 minutes on first run.\n")

# Install all packages from requirements.txt
!{sys.executable} -m pip install -r requirements.txt

# Download spacy model
print("\nDownloading Spacy English model...")
!{sys.executable} -m spacy download en_core_web_sm

# Download textblob corpora
print("\nDownloading TextBlob corpora...")
!{sys.executable} -m python -m textblob.download_corpora

print("\n✓ Installation complete!")
print("You can now run the rest of the cells.")

In [ ]:
# Import libraries
import requests
import json
import pandas as pd
import numpy as np
from pathlib import Path
import re
import time
from datetime import datetime
from collections import Counter

# NLP and Sentiment Analysis
import nltk
from nltk.corpus import stopwords
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Visualization
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Download NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('vader_lexicon', quiet=True)

# Initialize sentiment analyzer
vader_analyzer = SentimentIntensityAnalyzer()

# Set display options
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 100)

print("All libraries loaded successfully!")

In [ ]:
# ===== CONFIGURATION =====
# Insert your TROVE API key here
api_key = ''  # Get your key from https://trove.nla.gov.au/

# Animal species to analyze (you can add more)
animals = [
    'kangaroo',
    'koala',
    'dingo',
    'platypus',
    'wombat',
    'crocodile',
    'snake',
    'shark'
]

# Date range
start_year = 1900
end_year = 1950

# States to analyze (leave empty for all states)
# Options: 'ACT', 'New South Wales', 'Northern Territory', 'Queensland', 
#          'South Australia', 'Tasmania', 'Victoria', 'Western Australia'
states = []  # Empty list means all states

# Article types to search
# Options: 'Article', 'Advertising', 'Detailed lists, results, guides', 'Family Notices', 'Literature'
article_types = ['Article']  # Empty list means all types

# Maximum articles per animal (to avoid overwhelming API)
max_articles_per_animal = 500

# Minimum relevance score (5 is recommended)
min_relevance_score = 5

print(f"Configuration set: Analyzing {len(animals)} animals from {start_year} to {end_year}")

In [ ]:
# ===== TROVE API FUNCTIONS =====

def query_trove_animal(animal, api_key, start_year, end_year, states=None, article_types=None, max_results=500):
    """
    Query TROVE API for articles mentioning a specific animal.
    
    Args:
        animal: Animal species name to search for
        api_key: TROVE API key
        start_year: Starting year for search
        end_year: Ending year for search
        states: List of Australian states to filter by
        article_types: List of article types to filter by
        max_results: Maximum number of results to retrieve
    
    Returns:
        DataFrame containing article data
    """
    
    params = {
        'key': api_key,
        'zone': 'newspaper',
        'include': 'articleText',
        'n': 100,  # Results per page (max 100)
        'encoding': 'json',
        'bulkHarvest': 'false',
        'reclevel': 'brief',
        'sortby': 'relevance'
    }
    
    # Add optional filters
    if states:
        params['l-state'] = states
    if article_types:
        params['l-category'] = article_types
    
    # Build query
    params['q'] = f'{animal} date:[{start_year} TO {end_year}]'
    
    all_articles = []
    total_retrieved = 0
    
    print(f"Querying TROVE for '{animal}'...", end=" ")
    
    # Initial request to get total results
    response = requests.get('https://api.trove.nla.gov.au/v2/result', params=params)
    
    if response.status_code != 200:
        print(f"Error: API returned status code {response.status_code}")
        return pd.DataFrame()
    
    data = response.json()
    
    try:
        total_available = int(data['response']['zone'][0]['records']['total'])
        articles = data['response']['zone'][0]['records'].get('article', [])
    except (KeyError, IndexError):
        print("No results found.")
        return pd.DataFrame()
    
    all_articles.extend(articles)
    total_retrieved = len(articles)
    
    # Get additional pages if needed
    while total_retrieved < min(max_results, total_available) and 's' in params:
        params['s'] = f"*:{total_retrieved}"
        time.sleep(0.2)  # Rate limiting
        
        response = requests.get('https://api.trove.nla.gov.au/v2/result', params=params)
        if response.status_code != 200:
            break
            
        data = response.json()
        try:
            articles = data['response']['zone'][0]['records'].get('article', [])
            if not articles:
                break
            all_articles.extend(articles)
            total_retrieved += len(articles)
        except (KeyError, IndexError):
            break
    
    # Add one more pagination attempt
    if total_retrieved == 100 and total_retrieved < min(max_results, total_available):
        params['s'] = f"*:{total_retrieved}"
        time.sleep(0.2)
        response = requests.get('https://api.trove.nla.gov.au/v2/result', params=params)
        if response.status_code == 200:
            data = response.json()
            try:
                articles = data['response']['zone'][0]['records'].get('article', [])
                all_articles.extend(articles)
                total_retrieved += len(articles)
            except (KeyError, IndexError):
                pass
    
    print(f"Retrieved {total_retrieved} articles (total available: {total_available})")
    
    if not all_articles:
        return pd.DataFrame()
    
    # Convert to DataFrame
    df = pd.json_normalize(all_articles)
    
    # Add animal column
    df['animal'] = animal
    
    # Clean and process data
    if 'relevance.score' in df.columns:
        df['relevance'] = df['relevance.score'].astype('float')
    
    # Extract article text
    if 'articleText' in df.columns:
        df['article_text'] = df['articleText'].str.replace(r'<[^<>]*>', '', regex=True)
    
    # Convert date
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df['year'] = df['date'].dt.year
        df['month'] = df['date'].dt.month
    
    # Extract state if available
    if 'state' in df.columns:
        df['state'] = df['state']
    
    return df


def collect_all_animals_data(animals, api_key, start_year, end_year, **kwargs):
    """
    Collect data for multiple animals.
    """
    all_data = []
    
    for animal in animals:
        df = query_trove_animal(animal, api_key, start_year, end_year, **kwargs)
        if not df.empty:
            all_data.append(df)
        time.sleep(0.5)  # Be nice to the API
    
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        print(f"\nTotal articles collected: {len(combined_df)}")
        return combined_df
    else:
        print("No data collected.")
        return pd.DataFrame()

print("TROVE query functions loaded.")

In [ ]:
# ===== SENTIMENT ANALYSIS FUNCTIONS =====

def analyze_sentiment_textblob(text):
    """
    Analyze sentiment using TextBlob.
    Returns polarity (-1 to 1) and subjectivity (0 to 1).
    """
    if pd.isna(text) or not text:
        return 0, 0
    
    try:
        blob = TextBlob(str(text))
        return blob.sentiment.polarity, blob.sentiment.subjectivity
    except:
        return 0, 0


def analyze_sentiment_vader(text):
    """
    Analyze sentiment using VADER (better for social media/informal text).
    Returns compound score (-1 to 1).
    """
    if pd.isna(text) or not text:
        return {'neg': 0, 'neu': 0, 'pos': 0, 'compound': 0}
    
    try:
        scores = vader_analyzer.polarity_scores(str(text))
        return scores
    except:
        return {'neg': 0, 'neu': 0, 'pos': 0, 'compound': 0}


def categorize_sentiment(score):
    """
    Categorize sentiment score into positive, neutral, or negative.
    """
    if score > 0.1:
        return 'positive'
    elif score < -0.1:
        return 'negative'
    else:
        return 'neutral'


def add_sentiment_analysis(df):
    """
    Add sentiment analysis columns to the dataframe.
    """
    if 'article_text' not in df.columns:
        print("Warning: No article_text column found")
        return df
    
    print("Analyzing sentiment...")
    
    # TextBlob sentiment
    sentiments = df['article_text'].apply(analyze_sentiment_textblob)
    df['polarity'] = sentiments.apply(lambda x: x[0])
    df['subjectivity'] = sentiments.apply(lambda x: x[1])
    
    # VADER sentiment
    vader_scores = df['article_text'].apply(analyze_sentiment_vader)
    df['vader_negative'] = vader_scores.apply(lambda x: x['neg'])
    df['vader_neutral'] = vader_scores.apply(lambda x: x['neu'])
    df['vader_positive'] = vader_scores.apply(lambda x: x['pos'])
    df['vader_compound'] = vader_scores.apply(lambda x: x['compound'])
    
    # Categorize
    df['sentiment_category'] = df['vader_compound'].apply(categorize_sentiment)
    
    print("Sentiment analysis complete!")
    return df

print("Sentiment analysis functions loaded.")

In [ ]:
# ===== VISUALIZATION FUNCTIONS =====

def create_wordcloud(text, title, output_path=None):
    """
    Create and display a word cloud.
    """
    if not text or pd.isna(text):
        print(f"No text available for {title}")
        return
    
    # Create word cloud
    wc = WordCloud(
        width=1200,
        height=600,
        background_color='white',
        colormap='viridis',
        collocations=True,
        min_word_length=3,
        max_words=100
    )
    
    wc.generate(str(text))
    
    # Display
    plt.figure(figsize=(15, 8))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
    
    plt.show()


def create_sentiment_wordclouds(df, animal):
    """
    Create separate word clouds for positive and negative sentiment articles.
    """
    animal_df = df[df['animal'] == animal]
    
    # Positive articles
    positive_text = ' '.join(animal_df[animal_df['sentiment_category'] == 'positive']['article_text'].dropna())
    if positive_text:
        create_wordcloud(positive_text, f"Positive Sentiment Word Cloud - {animal.title()}")
    
    # Negative articles
    negative_text = ' '.join(animal_df[animal_df['sentiment_category'] == 'negative']['article_text'].dropna())
    if negative_text:
        create_wordcloud(negative_text, f"Negative Sentiment Word Cloud - {animal.title()}")


def plot_sentiment_over_time(df, animals=None):
    """
    Plot sentiment trends over time for specified animals.
    """
    if animals is None:
        animals = df['animal'].unique()
    
    # Group by year and animal
    yearly_sentiment = df.groupby(['year', 'animal']).agg({
        'vader_compound': 'mean',
        'polarity': 'mean',
        'article_text': 'count'
    }).reset_index()
    yearly_sentiment.columns = ['year', 'animal', 'avg_vader_score', 'avg_polarity', 'article_count']
    
    # Filter for specified animals
    yearly_sentiment = yearly_sentiment[yearly_sentiment['animal'].isin(animals)]
    
    # Create interactive plot
    fig = px.line(
        yearly_sentiment,
        x='year',
        y='avg_vader_score',
        color='animal',
        title='Sentiment Trends Over Time (VADER Compound Score)',
        labels={'avg_vader_score': 'Average Sentiment Score', 'year': 'Year', 'animal': 'Animal'},
        markers=True,
        hover_data=['article_count']
    )
    
    fig.add_hline(y=0, line_dash="dash", line_color="gray", annotation_text="Neutral")
    fig.update_layout(height=600, hovermode='x unified')
    fig.show()


def plot_sentiment_distribution(df):
    """
    Plot sentiment distribution for each animal.
    """
    # Count sentiment categories by animal
    sentiment_counts = df.groupby(['animal', 'sentiment_category']).size().reset_index(name='count')
    
    fig = px.bar(
        sentiment_counts,
        x='animal',
        y='count',
        color='sentiment_category',
        title='Sentiment Distribution by Animal',
        labels={'count': 'Number of Articles', 'animal': 'Animal', 'sentiment_category': 'Sentiment'},
        color_discrete_map={'positive': 'green', 'neutral': 'gray', 'negative': 'red'},
        barmode='group'
    )
    
    fig.update_layout(height=600)
    fig.show()


def create_sentiment_heatmap_by_state(df):
    """
    Create a heatmap showing average sentiment by animal and state.
    """
    if 'state' not in df.columns:
        print("No state information available for heatmap")
        return
    
    # Calculate average sentiment by animal and state
    heatmap_data = df.groupby(['animal', 'state'])['vader_compound'].mean().reset_index()
    heatmap_pivot = heatmap_data.pivot(index='animal', columns='state', values='vader_compound')
    
    # Create heatmap
    plt.figure(figsize=(14, 8))
    sns.heatmap(
        heatmap_pivot,
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        center=0,
        cbar_kws={'label': 'Average Sentiment Score'},
        linewidths=0.5
    )
    plt.title('Sentiment Heatmap by Animal and State', fontsize=16, fontweight='bold')
    plt.xlabel('State', fontsize=12)
    plt.ylabel('Animal', fontsize=12)
    plt.tight_layout()
    plt.show()


def create_temporal_heatmap(df, animal=None):
    """
    Create a heatmap showing sentiment over time (years vs months).
    """
    plot_df = df.copy()
    
    if animal:
        plot_df = plot_df[plot_df['animal'] == animal]
        title = f'Temporal Sentiment Heatmap - {animal.title()}'
    else:
        title = 'Temporal Sentiment Heatmap - All Animals'
    
    if plot_df.empty:
        print(f"No data available for {title}")
        return
    
    # Group by year and month
    temporal_data = plot_df.groupby(['year', 'month'])['vader_compound'].mean().reset_index()
    temporal_pivot = temporal_data.pivot(index='month', columns='year', values='vader_compound')
    
    # Create heatmap
    plt.figure(figsize=(16, 8))
    sns.heatmap(
        temporal_pivot,
        annot=False,
        cmap='RdYlGn',
        center=0,
        cbar_kws={'label': 'Average Sentiment Score'},
        linewidths=0.1
    )
    plt.title(title, fontsize=16, fontweight='bold')
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('Month', fontsize=12)
    plt.yticks(range(12), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                            'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
    plt.tight_layout()
    plt.show()

print("Visualization functions loaded.")

In [ ]:
# ===== COLLECT DATA FROM TROVE =====

# Collect data for all specified animals
df_animals = collect_all_animals_data(
    animals=animals,
    api_key=api_key,
    start_year=start_year,
    end_year=end_year,
    states=states if states else None,
    article_types=article_types if article_types else None,
    max_results=max_articles_per_animal
)

# Filter by relevance score if specified
if not df_animals.empty and 'relevance' in df_animals.columns:
    df_animals = df_animals[df_animals['relevance'] >= min_relevance_score]
    print(f"After filtering by relevance >= {min_relevance_score}: {len(df_animals)} articles")

# Display summary
if not df_animals.empty:
    print("\n=== Data Collection Summary ===")
    print(df_animals.groupby('animal').size().sort_values(ascending=False))
else:
    print("No data collected. Check your API key and search parameters.")

In [ ]:
# ===== PERFORM SENTIMENT ANALYSIS =====

if not df_animals.empty:
    df_animals = add_sentiment_analysis(df_animals)
    
    # Display sentiment summary
    print("\n=== Sentiment Analysis Summary ===")
    print("\nAverage sentiment by animal (VADER compound score):")
    print(df_animals.groupby('animal')['vader_compound'].mean().sort_values(ascending=False))
    
    print("\nSentiment category distribution:")
    print(df_animals.groupby(['animal', 'sentiment_category']).size().unstack(fill_value=0))
else:
    print("No data available for sentiment analysis.")

In [ ]:
# ===== SAVE RESULTS TO CSV =====

if not df_animals.empty:
    # Create output directory
    output_dir = Path(f"animal_sentiment_{start_year}_{end_year}")
    output_dir.mkdir(exist_ok=True)
    
    # Save complete dataset
    output_file = output_dir / f"all_animals_sentiment_{start_year}_{end_year}.csv"
    df_animals.to_csv(output_file, index=False)
    print(f"\nComplete dataset saved to: {output_file}")
    
    # Save individual animal files
    for animal in df_animals['animal'].unique():
        animal_df = df_animals[df_animals['animal'] == animal]
        animal_file = output_dir / f"{animal}_sentiment_{start_year}_{end_year}.csv"
        animal_df.to_csv(animal_file, index=False)
        print(f"Saved {animal} data to: {animal_file}")
    
    # Save summary statistics
    summary = df_animals.groupby('animal').agg({
        'vader_compound': ['mean', 'std', 'min', 'max'],
        'polarity': ['mean', 'std'],
        'article_text': 'count'
    }).round(4)
    summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
    summary_file = output_dir / f"summary_statistics_{start_year}_{end_year}.csv"
    summary.to_csv(summary_file)
    print(f"\nSummary statistics saved to: {summary_file}")
    
    print(f"\nAll files saved to directory: {output_dir}")
else:
    print("No data to save.")

In [ ]:
# ===== VISUALIZATION: Sentiment Distribution =====

if not df_animals.empty:
    plot_sentiment_distribution(df_animals)
else:
    print("No data available for visualization.")

In [ ]:
# ===== VISUALIZATION: Sentiment Trends Over Time =====

if not df_animals.empty:
    plot_sentiment_over_time(df_animals)
else:
    print("No data available for visualization.")

In [ ]:
# ===== VISUALIZATION: Sentiment Heatmap by State =====

if not df_animals.empty:
    create_sentiment_heatmap_by_state(df_animals)
else:
    print("No data available for visualization.")

In [ ]:
# ===== VISUALIZATION: Temporal Heatmap (Year x Month) =====

if not df_animals.empty:
    # Create heatmap for all animals combined
    create_temporal_heatmap(df_animals)
    
    # Create individual heatmaps for animals with enough data
    for animal in df_animals['animal'].unique():
        animal_count = len(df_animals[df_animals['animal'] == animal])
        if animal_count >= 20:  # Only create if enough data points
            create_temporal_heatmap(df_animals, animal=animal)
else:
    print("No data available for visualization.")

In [ ]:
# ===== VISUALIZATION: Word Clouds =====

if not df_animals.empty:
    for animal in df_animals['animal'].unique():
        print(f"\n{'='*60}")
        print(f"Word Clouds for: {animal.upper()}")
        print(f"{'='*60}\n")
        
        # Overall word cloud
        animal_text = ' '.join(df_animals[df_animals['animal'] == animal]['article_text'].dropna())
        if animal_text:
            create_wordcloud(animal_text, f"Overall Word Cloud - {animal.title()}")
        
        # Sentiment-specific word clouds
        create_sentiment_wordclouds(df_animals, animal)
else:
    print("No data available for word clouds.")

In [ ]:
# ===== DETAILED ANALYSIS: Top Articles by Sentiment =====

if not df_animals.empty:
    print("\n" + "="*80)
    print("MOST POSITIVE ARTICLES")
    print("="*80)
    
    top_positive = df_animals.nlargest(5, 'vader_compound')[[
        'animal', 'heading', 'date', 'vader_compound', 'title.value'
    ]]
    
    for idx, row in top_positive.iterrows():
        print(f"\nAnimal: {row['animal'].upper()}")
        print(f"Sentiment Score: {row['vader_compound']:.3f}")
        print(f"Date: {row['date']}")
        if 'title.value' in row:
            print(f"Newspaper: {row['title.value']}")
        if 'heading' in row:
            print(f"Headline: {row['heading']}")
        print("-" * 80)
    
    print("\n" + "="*80)
    print("MOST NEGATIVE ARTICLES")
    print("="*80)
    
    top_negative = df_animals.nsmallest(5, 'vader_compound')[[
        'animal', 'heading', 'date', 'vader_compound', 'title.value'
    ]]
    
    for idx, row in top_negative.iterrows():
        print(f"\nAnimal: {row['animal'].upper()}")
        print(f"Sentiment Score: {row['vader_compound']:.3f}")
        print(f"Date: {row['date']}")
        if 'title.value' in row:
            print(f"Newspaper: {row['title.value']}")
        if 'heading' in row:
            print(f"Headline: {row['heading']}")
        print("-" * 80)
else:
    print("No data available for detailed analysis.")

In [ ]:
# ===== COMPARATIVE ANALYSIS =====

if not df_animals.empty:
    # Box plot comparing sentiment distributions
    fig = px.box(
        df_animals,
        x='animal',
        y='vader_compound',
        color='animal',
        title='Sentiment Distribution Comparison Across Animals',
        labels={'vader_compound': 'Sentiment Score (VADER)', 'animal': 'Animal'},
        points='outliers'
    )
    fig.add_hline(y=0, line_dash="dash", line_color="gray", annotation_text="Neutral")
    fig.update_layout(height=600, showlegend=False)
    fig.show()
    
    # Violin plot for detailed distribution
    fig = px.violin(
        df_animals,
        x='animal',
        y='vader_compound',
        color='animal',
        title='Sentiment Distribution (Violin Plot)',
        labels={'vader_compound': 'Sentiment Score (VADER)', 'animal': 'Animal'},
        box=True,
        points='outliers'
    )
    fig.add_hline(y=0, line_dash="dash", line_color="gray", annotation_text="Neutral")
    fig.update_layout(height=600, showlegend=False)
    fig.show()
else:
    print("No data available for comparative analysis.")

## Analysis Complete!

This notebook has:
1. Queried the TROVE API for articles about specified animals
2. Performed machine learning-based sentiment analysis using both TextBlob and VADER
3. Generated multiple visualizations:
   - Word clouds (overall and sentiment-specific)
   - Sentiment trend graphs over time
   - Geographic heat maps by state
   - Temporal heat maps (year x month)
   - Comparative distribution plots
4. Saved all results to CSV files

You can now explore the data further or modify the parameters to analyze different animals or time periods!